# Simple MCP demo
- Send a query to LLM, says it doesn't know
- Give it a tool to help, and it knows!<br />&nbsp;<br />

- Model Context Protocol (MCP) is like a USB standard for LLM/tool integration: it lets you plug 3rd party tools into your AI, letting e.g. a desktop client make Plotly charts or request additional info from Wikipedia. <br />&nbsp;<br />

- LLMs on their owns are like librarians, you can talk to them, they can understand you at some level and give you information back. With MCP and tools, they gain access to more information, can take actions, potentially including internal network apps and data, and can become more like personal assistants.<br />&nbsp;<br />

- MCP has 3 components:
  - An MCP client which initiates a conversation and requests to the LLM and MCP server. MCP client is for example a chat client (Claude Desktop is great) or an IDE (Cursor, Windsurf) that is asking an LLM for help with code
  - An MCP server which provides tools for a purpose - A Fetch or Playwright tool to browse the Web, or a Context7 tool to look up Python module documentation in a vector DB
  - An LLM which supports tool use - any major LLM that follows the OpenAI tool use standard <br />&nbsp;<br />

- MCP Flow:
  - User launches MCP client, it connects to MCP server(s) per its configuration.
  - MCP client asks MCP server to list the tools it offers to the client.
  - MCP client can prompt the LLM, providing a list of available tools (calling signatures, and semantic descriptions of when to call them).
  - LLM responds to prompt. If, based on the prompt and the available tools, a tool would be the best way to answer the question, LLM will respond with a tool call request and the parameter values for the calling signature.
  - MCP client then calls the tools using the provided signature, adds the output from the tool to the conversation, and calls the LLM again with the updated conversation.
  - LLM may respond with further tool call requests, or provide a response.
  - That's mostly it. Besides executing tool calls, servers can also provide static resources for the client like docs, reference prompts, see the [docs on the home page](https://modelcontextprotocol.io/overview) <br />&nbsp;<br />

- &gt; 10,000 MCP servers available 
  - [MCP Market leaderboard (by GitHub stars)](https://mcpmarket.com/leaderboards)
  - [PulseMCP directory (by downloads)](https://www.pulsemcp.com/servers?sort=popular-30-days-desc)
  - [LobeHub](https://lobehub.com/mcp)
  - [Glama](https://glama.ai/mcp/servers)<br />&nbsp;<br />

- Without even coding, you can connect a client like Claude Desktop to them via configurations, and you get reasoning models + deep research + actions, which can be a force multiplier for analysts and knowledge workers.<br />&nbsp;<br />

- In the example below, we make an MCP server in a few lines of code to answer Monty Python's most famous question, and insert some pdb breakpoints so we can step through the flow. <br />&nbsp;<br />


- More info:
  - [Anthropic MCP Announcement](https://www.anthropic.com/news/model-context-protocol)
  - [Anthropic YouTube talk](https://www.youtube.com/watch?v=kQmXtrmQ5Zg)
  - [Model Context Protocol home page on GitHub](https://github.com/modelcontextprotocol)
  - [Composio intro](https://composio.dev/blog/what-is-model-context-protocol-mcp-explained)<br />&nbsp;<br />
      
      
  

![!image.png](q5AltSX5E3TfLsmtZp5jjLMU5U.png)


# Example Code

In [91]:
import sys
import os
import dotenv
import re
from datetime import datetime, timedelta
import time
from typing import Dict, Any, Optional, Annotated
from urllib.parse import urljoin, urlparse

import asyncio
import nest_asyncio

from contextlib import AsyncExitStack
import mcp
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

import anthropic
from anthropic import Anthropic
import pdb


In [92]:
# load secrets from .env including API keys
dotenv.load_dotenv()

# enable asyncio in jupyter notebook
nest_asyncio.apply()

# Initialize plotly for Jupyter
# init_notebook_mode(connected=True)


In [93]:
# test anthropic
client = anthropic.Anthropic()

# https://docs.anthropic.com/en/docs/about-claude/models/overview
anthropic_models = [
    "claude-opus-4-20250514",
    "claude-sonnet-4-20250514",
    "claude-3-5-haiku-20241022",
]

print("Available Claude models:")
print("\n".join(anthropic_models))
print()

# Try making a simple completion request to each:

message = "what is the airspeed velocity of an unladen swallow"
for model in anthropic_models:
    try:
        response = client.messages.create(
            model=model,
            max_tokens=200,
            messages=[{"role": "user", "content": message}]
        )
        print(f"✓ {model}")
        print(response.content[0].text)
        print()
    except Exception as e:
        print(f"✗ {model} - error: {str(e)}")

Available Claude models:
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-5-haiku-20241022

✓ claude-opus-4-20250514
The famous question from Monty Python! The answer depends on whether you mean an African or European swallow.

According to actual estimates by fans and ornithologists who've analyzed this question:

- **European swallow**: roughly 20.1 mph (32.4 km/h)
- **African swallow**: roughly 24 mph (38.6 km/h)

These are based on the Strouhal number and estimates of wing beat frequency and amplitude for these birds.

Of course, in the context of *Monty Python and the Holy Grail*, the proper response is: "What do you mean? An African or European swallow?" - at which point the bridge keeper realizes he doesn't know and is cast into the Gorge of Eternal Peril.

✓ claude-sonnet-4-20250514
Ah, a classic Monty Python reference! 

The proper response is: "What do you mean? An African or European swallow?"

But if you want actual numbers:
- **European swallow** (barn swallow): ro

In [94]:
# test openai
from openai import OpenAI

client = OpenAI()
# https://platform.openai.com/docs/models
openai_models = [
    "gpt-4o",
    "gpt-4o-mini",
    "gpt-4.1",
    "gpt-4.1-mini",
    "o3"
]
print("Available OpenAI models:")
print("\n".join(openai_models))
print()

# Try making a simple completion request to each:
message = "what is the airspeed velocity of an unladen swallow"
for model in openai_models:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": message}]
        )
        print(f"✓ {model}")
        print(response.choices[0].message.content)
        print()
    except Exception as e:
        print(f"✗ {model} - error: {str(e)}")



Available OpenAI models:
gpt-4o
gpt-4o-mini
gpt-4.1
gpt-4.1-mini
o3

✓ gpt-4o
The question about the airspeed velocity of an unladen swallow is a humorous reference to the film "Monty Python and the Holy Grail." In a more serious context, the average cruising airspeed of an unladen European Swallow (Hirundo rustica) is estimated to be around 11 meters per second, or 24 miles per hour. However, it's important to note that this information is specific to the European Swallow, and the speeds can vary based on species and environmental conditions.

✓ gpt-4o-mini
The airspeed velocity of an unladen swallow is a humorous question popularized by the film "Monty Python and the Holy Grail." In a more scientific context, estimates suggest that the average cruising airspeed of a European swallow (Hirundo rustica) is about 11 meters per second, or roughly 24 miles per hour. However, it's important to note that this is a simplified answer, and actual speeds can vary based on various factors such as

see swallow_server.py
```
"""
This module swallow_server.py implements a simple MCP server using FastMCP,
providing a tool unladen_swallow_airspeed, returns a string based on input swallow type
"""
from mcp.server.fastmcp import FastMCP
from pydantic import Field, BaseModel

# return schema
class SwallowSpeed(BaseModel):
    speed: str
    unit: str
    swallow_type: str

mcp = FastMCP("swallow-server")

@mcp.tool()
def unladen_swallow_airspeed(
    swallow_type: str = Field(description="Type of swallow: 'african' or 'european'")
) -> SwallowSpeed:
    """Provides the airspeed velocity of an unladen swallow."""
    stype = swallow_type.strip().lower()
    if stype == 'african':
        return SwallowSpeed(speed="31.1415926", unit="km/h", swallow_type="african")
    elif stype == 'european':
        return SwallowSpeed(speed="27.1828", unit="km/h", swallow_type="european")
    else:
        return SwallowSpeed(speed="I don't know!", unit="", swallow_type=stype)


def main():
    mcp.run()


if __name__ == "__main__":
    main()
```

## Test swallow_server.py using MCP Inspector
- `$ mcp dev swallow_server.py`
- click 'connect'
- click 'tools'
- click 'unladen_swallow_airspeed' tool
- enter parameters

![MCP Inspector Image](swallow_server.png)

In [95]:
MODEL = 'gpt-4.1-mini'

class MCPClient:
    """An MCP client adapted to run in a Jupyter notebook.
    """
    def __init__(self, model=MODEL):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.model=model
        if model in anthropic_models:
            self.vendor ='Anthropic'
            self.llm = Anthropic()
        elif model in openai_models:
            self.vendor = 'OpenAI'
            self.llm = OpenAI()
        else:
            print(f"bad model {model}, try again")
        self.tools = {}
        self.tools_reverse = {}

    def connect_to_server(self, server_script_path: str):
        """Connect to an MCP server and list its tools."""
        print(f"Connecting to server: {server_script_path}...")
        is_python = server_script_path.endswith('.py')
        if not is_python:
            raise ValueError("Server script must be a .py file")

        server_params = StdioServerParameters(
            command=sys.executable,  # Use the same python executable
            args=[server_script_path],
            env=None
        )

        pdb.set_trace()
        # print(server_params)
        response = asyncio.run(self.async_connect_to_server(server_params))
        # print(response)
        self.tools[server_script_path] = response.tools
        reverse_tool_dict = {tool.name: server_script_path for tool in response.tools}
        self.tools_reverse = {**self.tools_reverse, **reverse_tool_dict}
        print("\nConnection successful!")
        print("Available tools:", [(tool.name, tool.description, tool.inputSchema) for tool in self.tools[server_script_path]])

    async def async_connect_to_server(self, server_params):

        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))

        await self.session.initialize()
        response = await self.session.list_tools()
        return response

    def process_query(self, query: str) -> str:
        # TODO: implement process_query_openai, call based on self.vendor
        if self.vendor == "Anthropic":
            return self.process_query_anthropic(query)
        elif self.vendor == "OpenAI":
            return self.process_query_openai(query)
        
        else:
            return (f"not implemented")


    def process_query_anthropic(self, query: str) -> str:
        """Process a query using LLM and the available tools."""
        
        if not self.session:
            return "Error: Not connected to a server. Please run connect_to_server first."

        pdb.set_trace()
        messages = [{"role": "user", "content": query}]
        available_tools = [{
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.inputSchema
        } for server in self.tools.values() for tool in server]
        print(f"Sending query to {self.model}...")
        response = self.llm.messages.create(
            model=self.model, 
            max_tokens=1024,
            messages=messages,
            tools=available_tools
        )

        final_text = []
        for content in response.content:
            if content.type == 'text':
                final_text.append(content.text)
            elif content.type == 'tool_use':
                tool_name = content.name
                tool_args = content.input
                print(f"{self.model} requested to use tool: {tool_name} with arguments: {tool_args}")

                result = asyncio.run(self.session.call_tool(tool_name, tool_args))
                print(f"Received result from MCP tool: {result.content} ")

                # Create the tool result content block
                tool_result_content = {
                    "type": "tool_result",
                    "tool_use_id": content.id,
                    "content": str(result.content) # Ensure content is a string
                }

                # Append the original assistant message and the tool result
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": [tool_result_content]})

                # Get next response from LLM
                print(f"Tool result: {tool_result_content}")
                print(f"Sending tool result back to {self.model}...")
                follow_up_response = self.llm.messages.create(
                    model=self.model,
                    max_tokens=1024,
                    messages=messages,
                )
                for follow_up_content in follow_up_response.content:
                    if follow_up_content.type == 'text':
                        final_text.append(follow_up_content.text)  

        return "\n".join(final_text)


    def process_query_openai(self, query: str) -> str:
        """Process a query using LLM and the available tools."""
        
        if not self.session:
            return "Error: Not connected to a server. Please run connect_to_server first."

        pdb.set_trace()
        messages = [{"role": "user", "content": query}]
        available_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                }
                for server in self.tools.values() for tool in server]
        print(f"Sending query to {self.model}...")
        response = self.llm.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=available_tools,
            tool_choice="auto",  # auto = let the model decide whether to call a tool
        )
        response_message = response.choices[0].message

        # Check if the model decided to call a tool
        if response_message.tool_calls:
            tool_call = response_message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            print(f"{self.model} requested to use tool: {tool_name} with arguments: {tool_args}")
            result = asyncio.run(self.session.call_tool(tool_name, tool_args))
            print(f"Received result from MCP tool: {result.content} ")

            # Append the original assistant message and the tool result
            messages.append({
                "role": "assistant",
                "content": response_message.content,  # This might be None for tool calls
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    }
                ]
            })
            # Add tool result
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_name,
                "content": json.dumps(result.structuredContent)
            })
            # Send the messages with the tool's output back to the model
            second_response = self.llm.chat.completions.create(
                model=self.model,
                messages=messages,
            )
            second_response_message = second_response.choices[0].message
            return second_response_message.content
        else:
            return response_message.content

        return "\n".join(final_text)

    def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Client Started!")
        print("Type your queries or 'quit' to exit.")

        while True:
            try:
                query = input("\nQuery: ").strip()

                if query.lower() == 'quit':
                    break

                response = self.process_query(query)
                print("\n" + response)

            except Exception as e:
                print(f"\nError: {str(e)}")

    def cleanup(self):
        """Clean up resources and close the server connection."""
        print("Cleaning up resources...")
        asyncio.run(self.exit_stack.aclose())
        print("Cleanup complete.")


In [96]:
def connect():
    client = MCPClient(MODEL)
    client.connect_to_server('swallow_server.py')
    return client

# Run the connection and keep the client object
# This will block until the connection is established.
client = connect()


Connecting to server: swallow_server.py...
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_39327/467637067.py(36)connect_to_server()
     34         pdb.set_trace()
     35         # print(server_params)
---> 36         response = asyncio.run(self.async_connect_to_server(server_params))
     37         # print(response)
     38         self.tools[server_script_path] = response.tools

ipdb> c

Connection successful!
Available tools: [('unladen_swallow_airspeed', 'Provides the airspeed velocity of an unladen swallow.', {'properties': {'swallow_type': {'description': "Type of swallow: 'african' or 'european'", 'title': 'Swallow Type', 'type': 'string'}}, 'required': ['swallow_type'], 'title': 'unladen_swallow_airspeedArguments', 'type': 'object'})]


In [97]:
def run_query(query):
    response = client.process_query(query)
    print("\n--- LLM's Response ---")
    print(response)
    print("-------------------------")


In [98]:
# Run a query
query = "What is the airspeed velocity of an unladen african swallow?"
run_query(query)


> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_39327/467637067.py(131)process_query_openai()
    129 
    130         pdb.set_trace()
--> 131         messages = [{"role": "user", "content": query}]
    132         available_tools = [
    133                 {

ipdb> c
Sending query to gpt-4.1-mini...
gpt-4.1-mini requested to use tool: unladen_swallow_airspeed with arguments: {'swallow_type': 'african'}
Received result from MCP tool: [TextContent(type='text', text='{\n  "speed": "31.1415926",\n  "unit": "km/h",\n  "swallow_type": "african"\n}', annotations=None, meta=None)] 

--- LLM's Response ---
The airspeed velocity of an unladen African swallow is approximately 31.1 km/h.
-------------------------


In [99]:
# Run a chat loop
client.chat_loop()



MCP Client Started!
Type your queries or 'quit' to exit.

Query: What is the airspeed velocity of an unladen american swallow?
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_39327/467637067.py(131)process_query_openai()
    129 
    130         pdb.set_trace()
--> 131         messages = [{"role": "user", "content": query}]
    132         available_tools = [
    133                 {

ipdb> c
Sending query to gpt-4.1-mini...

The airspeed velocity of an unladen swallow is typically referenced to either the African or European swallow. The "American swallow" is not a standard category for this measurement in the classic context. Would you like the airspeed velocity of an African swallow, a European swallow, or information on something else?

Query: quit
